## Carga de datos en una base de datos MySQL

### Objetivo
Este notebook carga los datasets procesados en una base de datos MySQL relacional, estableciendo la estructura de **modelo estrella** necesaria para el análisis de resiliencia empresarial.

### Estructura de la base de datos
- **Base de datos**: `ipc_analisis_empresarial`
- **Tablas de dimensiones**: `territorio`, `tiempo`, `tipo_medida`, `sectores_ipc`
- **Tablas de hechos**: `ipc`, `empresas_constituidas`, `empresas_disueltas`

### Metodología
1. **Conexión** a MySQL mediante `sqlalchemy` usando variables de entorno (`.env`).
2. **Creación** de la base de datos si no existe.
3. **Carga secuencial** de tablas dimensión y asignación de claves primarias.
4. **Carga de tablas de hechos** con asignación de claves primarias (autoincremental para `ipc`).
5. **Establecimiento de relaciones** — Creación de claves foráneas que vinculan las tablas de hechos con sus dimensiones correspondientes.

In [10]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd

 # Agrega la carpeta raíz al path de Python para importar módulos de src/
sys.path.append(os.path.abspath(os.path.join('..')))

# Importación de módulos de carga de datos
import src.load.load_db as db

# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)


# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())
from dotenv import load_dotenv

# Cargar y comprobar variables
load_dotenv("../.env", override=True)

Directorio actual: c:\Users\fabih\Desktop\CREATOR\data\spain-business-resilience-analytics\spain-business-resilience-analytics\notebooks


True

# Cargamos los datasets a DataFrames

In [5]:
df_empr_const = pd.read_csv('../files/data_processed/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_processed/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_processed/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_processed/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_processed/territorio.csv')
df_tiempo = pd.read_csv('../files/data_processed/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_processed/tipo_medida.csv')

### Limpieza inicial
Elimina las tablas existentes en orden de dependencia inversa para evitar conflictos con claves foráneas. Permite una recarga completa desde cero.

In [6]:
db.drop_all_tables("ipc_analisis_empresarial", [
    "empresas_disueltas",
    "empresas_constituidas",
    "ipc",
    "tipo_medida",
    "sectores_ipc",
    "tiempo",
    "territorio",
])

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

# Cargamos los Dataframes a la base de datos

In [ ]:
# Obtenemos la cadena de conexión sin seleccionar BD aún
conn_str = db.get_connection_string(db_name=None)
# Mostramos solo el protocolo por seguridad (no la contraseña)
print(f"✅ Conexión configurada correctamente: {conn_str.split(':')[0]}@****")

✅ Conexión configurada correctamente: mysql+pymysql@****


In [9]:
db.create_database_if_not_exists("ipc_analisis_empresarial")

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# Carga dimensión territorio
db.load_dataframe_to_mysql(df_territorio, "territorio", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a territorio
db.set_primary_key("territorio", "id_territorio", "ipc_analisis_empresarial")

In [ ]:
# Carga dimensión tiempo
db.load_dataframe_to_mysql(df_tiempo, "tiempo", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a tiempo
db.set_primary_key("tiempo", "id_tiempo", "ipc_analisis_empresarial")

In [ ]:
# Carga dimensión tipo_medida
db.load_dataframe_to_mysql(df_tipo_medida, "tipo_medida", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a tipo_medida
db.set_primary_key("tipo_medida", "id_medida", "ipc_analisis_empresarial")

In [ ]:
# Carga dimensión sectores_ipc
db.load_dataframe_to_mysql(df_sectores_ipc, "sectores_ipc", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a sectores_ipc
db.set_primary_key("sectores_ipc", "id_sector", "ipc_analisis_empresarial")

Hechos — tabla ipc (con autoincremental + FKs)

In [ ]:
# Carga tabla de hechos ipc (todavía sin PK propia)
db.load_dataframe_to_mysql(df_ipc, "ipc", "ipc_analisis_empresarial")

In [ ]:
# ipc usa clave compuesta; agregamos id_ipc AUTO_INCREMENT como PK
db.add_autoincrement_id("ipc", "ipc_analisis_empresarial")

In [ ]:
# Relaciona ipc con sus 4 dimensiones: territorio, tiempo, tipo_medida, sectores_ipc
db.set_foreign_keys(
    fact_table="ipc",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},
        {"fk_column": "id_medida", "dimension_table": "tipo_medida"},
        {"fk_column": "id_sector", "dimension_table": "sectores_ipc"},
    ],
    db_name="ipc_analisis_empresarial"
)

Hechos — empresas_constituidas

In [ ]:
# Carga tabla de hechos empresas_constituidas
db.load_dataframe_to_mysql(df_empr_const, "empresas_constituidas", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a empresas_constituidas
db.set_primary_key("empresas_constituidas", "id_const", "ipc_analisis_empresarial")

In [ ]:
# Relaciona empresas_constituidas con territorio y tiempo
db.set_foreign_keys(
    fact_table="empresas_constituidas",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},

    ],
    db_name="ipc_analisis_empresarial"
)

Hechos — empresas_disueltas

In [ ]:
# Carga tabla de hechos empresas_disueltas
db.load_dataframe_to_mysql(df_empr_dis, "empresas_disueltas", "ipc_analisis_empresarial")

In [ ]:
# Asigna PK a empresas_disueltas
db.set_primary_key("empresas_disueltas", "id_dis", "ipc_analisis_empresarial")

In [ ]:
# Relaciona empresas_disueltas con territorio y tiempo
db.set_foreign_keys(
    fact_table="empresas_disueltas",
    relations=[
        {"fk_column": "id_territorio", "dimension_table": "territorio"},
        {"fk_column": "id_tiempo", "dimension_table": "tiempo"},

    ],
    db_name="ipc_analisis_empresarial"
)